## Data Ingestion
Here we will load some PDFs, perform data processing, generate chunks and eventually create vector embeddings and store in the Vector DB. We will also explore **LangChain Document** data structure which is used to save data using some format including **page_content** and **metadata**.
- **page_content** is the actual page content
- **metadata** is the information about the document.
<br/>
<br/>
**LangChain Document Loader**
LangChain include various loaders to load the document of that specific type.
- PDFLoader
- CSVLoader
- WebBasedLoader
- DirectoryLoader
are some of the **loaders** which **output** in the **document structure**.


In [1]:
from langchain_core.documents import Document

In [4]:
# Create a test doc for experiment purposes.
test_doc = Document(page_content="This is a simple text content we are using as test purpose", metadata={
    "source": "online",
    "pages": 1,
    "publisher": "NextWave",
    "created": "2025-12-22"
})
test_doc

Document(metadata={'source': 'online', 'pages': 1, 'publisher': 'NextWave', 'created': '2025-12-22'}, page_content='This is a simple text content we are using as test purpose')

#### Create Document

In [6]:
# Create Document
import os
sample_texts = {
    "data/python_intro.txt": """Python Programming Introduction.
    Python is a high-level, general-purpose programming language known for its simplicity, readability, and versatility.
    Key Features:
- Readability: Python's design emphasizes clean, English-like syntax and uses indentation rather than curly brackets to define code blocks, which makes it easier to learn and write programs with fewer lines of code than many other languages.
- Interpreted Language: Code is executed line by line at runtime, which simplifies debugging and speeds up the edit-test-debug cycle.
- Dynamically Typed: Developers do not need to declare variable types explicitly; Python determines them during runtime, offering flexibility.
- Extensive Libraries: Python includes a large standard library and a vast ecosystem of third-party packages available through the Python Package Index (PyPI), which provide pre-written code for a wide range of tasks.
- Multiple Paradigms: It supports various programming styles, including procedural, object-oriented, and functional programming.
- Platform Independent: Python code can run on different operating systems like Windows, macOS, and Linux without needing major changes
""",
"data/machine_learning.txt": """Machine Learning Introduction
Machine learning is a subset of artificial intelligence (AI) where computers detect patterns in massive datasets to make predictions, decisions, or inferences without explicit programming. It works by using statistical algorithms to train models on data, allowing them to generalize and improve performance on new, unseen information. Key benefits include automation of complex tasks, rapid data analysis, and enhanced decision-making in fields like healthcare, finance, and technology. Key concepts include supervised learning (labeled data), unsupervised learning (unlabeled data), reinforcement learning (trial-and-error), and neural networks. Limitations involve the need for large, high-quality data, potential biases in training data, and the risk of overfitting
Types of Machine Learning:
- Supervised Learning: Models are trained on labeled data to predict outputs, such as linear regression (predicting values) or classification (labeling categories).
- Unsupervised Learning: Algorithms analyze unlabeled data to discover hidden structures, such as clustering similar items together.
- Reinforcement Learning: Agents learn to achieve goals through trial-and-error, receiving feedback (rewards or penalties) for their actions.
- Neural Networks/Deep Learning: Algorithms inspired by the human brain used for complex tasks like image and speech recognition.
"""
}
for filepath, content in sample_texts.items():
    with open(filepath, 'w', encoding='utf-8') as file:
        file.write(content)

### Document Loader

#### Text Loader

In [ ]:
# Read using Document Loaders
from langchain_community.document_loaders import TextLoader
loader = TextLoader("data/python_intro.txt", encoding="utf-8")
document = loader.load()
document

[Document(metadata={'source': 'data/python_intro.txt'}, page_content="Python Programming Introduction.\n    Python is a high-level, general-purpose programming language known for its simplicity, readability, and versatility.\n    Key Features:\n- Readability: Python's design emphasizes clean, English-like syntax and uses indentation rather than curly brackets to define code blocks, which makes it easier to learn and write programs with fewer lines of code than many other languages.\n- Interpreted Language: Code is executed line by line at runtime, which simplifies debugging and speeds up the edit-test-debug cycle.\n- Dynamically Typed: Developers do not need to declare variable types explicitly; Python determines them during runtime, offering flexibility.\n- Extensive Libraries: Python includes a large standard library and a vast ecosystem of third-party packages available through the Python Package Index (PyPI), which provide pre-written code for a wide range of tasks.\n- Multiple Par

In [10]:
# Show the content
document[0].page_content

"Python Programming Introduction.\n    Python is a high-level, general-purpose programming language known for its simplicity, readability, and versatility.\n    Key Features:\n- Readability: Python's design emphasizes clean, English-like syntax and uses indentation rather than curly brackets to define code blocks, which makes it easier to learn and write programs with fewer lines of code than many other languages.\n- Interpreted Language: Code is executed line by line at runtime, which simplifies debugging and speeds up the edit-test-debug cycle.\n- Dynamically Typed: Developers do not need to declare variable types explicitly; Python determines them during runtime, offering flexibility.\n- Extensive Libraries: Python includes a large standard library and a vast ecosystem of third-party packages available through the Python Package Index (PyPI), which provide pre-written code for a wide range of tasks.\n- Multiple Paradigms: It supports various programming styles, including procedural,

#### Directory Loader
Loads all the files within a directory matching the pattern


In [ ]:
from langchain_community.document_loaders import DirectoryLoader
dir_loaders = DirectoryLoader(path="data",
                              glob="*.txt", # pattern to match
                              loader_cls=TextLoader,
                              loader_kwargs={"encoding": "utf-8"},
                              show_progress=True
                              )

documents = dir_loaders.load()
print(documents)

100%|██████████| 2/2 [00:00<00:00, 1214.51it/s]

[Document(metadata={'source': 'data/python_intro.txt'}, page_content="Python Programming Introduction.\n    Python is a high-level, general-purpose programming language known for its simplicity, readability, and versatility.\n    Key Features:\n- Readability: Python's design emphasizes clean, English-like syntax and uses indentation rather than curly brackets to define code blocks, which makes it easier to learn and write programs with fewer lines of code than many other languages.\n- Interpreted Language: Code is executed line by line at runtime, which simplifies debugging and speeds up the edit-test-debug cycle.\n- Dynamically Typed: Developers do not need to declare variable types explicitly; Python determines them during runtime, offering flexibility.\n- Extensive Libraries: Python includes a large standard library and a vast ecosystem of third-party packages available through the Python Package Index (PyPI), which provide pre-written code for a wide range of tasks.\n- Multiple Par

In [18]:
### Read all PDF inside the directory
from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pathlib import Path

def document_processor(directory_path: str):
    """
    Process documents within the file_directory
    """
    documents = []
    dir = Path(directory_path)

    def process_pdf(pdf):
        print(f"Processing: {pdf.name}")
        try:
            loader = PyPDFLoader(str(pdf))
            document = loader.load()

            # Add source information to metadata
            for doc in document:
                doc.metadata['source_file'] = pdf.name
                doc.metadata['file_type'] = 'pdf'
            
            documents.extend(document)
            print(f"Loaded {len(document)} pages")
        except Exception as e:
            print(f"Error processing {pdf.name} file. Error: {e}")
                
    # Find all pdf files within the dir.
    pdf_files = list(dir.glob("**/*.pdf"))
    print(f"Found {len(pdf_files)} PDF files to process.")
    for pdf in pdf_files:
        process_pdf(pdf)

    return documents

In [12]:
!pip install pypdf

In [19]:
all_documents = document_processor(directory_path="./data/pdf")

Found 3 PDF files to process.
Processing: What Makes a Leader - Daniel Goleman HBR 2004_b6e637.pdf
Loaded 12 pages
Processing: The_right_way_to_hold_people_accountable_665076_9724ae.pdf
Loaded 4 pages
Processing: The 4 Disciplines of Execution.pdf
Loaded 8 pages


### Chunking
Once we are done with data ingestion, we start with Chunking

In [21]:
# Test splitting into chunks
def create_chunks(documents, chunk_size=1000, chunk_overlap=200):
    """ Split document into smaller chunks for getting better Embeddings. """
    text_splitter = RecursiveCharacterTextSplitter(
        separators=["\n\n", "\n", " ", "\t"],
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,        
    )
    split_docs = text_splitter.split_documents(documents)
    print(f"Split {len(documents)} documents into {len(split_docs)} chunks")

    # Show example of a chunk
    if split_docs:
        print(f"\nExample Chunk")
        print(f"Content: {split_docs[0].page_content[:200]}...")
        print(f"Metadata: {split_docs[0].metadata}...")
    return split_docs

In [22]:
chunks = create_chunks(all_documents)

Split 24 documents into 122 chunks

Example Chunk
Content: www.hbr.org
 
B
 
EST
 
 
 
OF
 
 HBR 1998
 
What Makes a Leader?
 
by Daniel Goleman
 
•
 
Included with this full-text Harvard Business Review article:
The Idea in Brief— the core idea
The Idea in P...
Metadata: {'producer': 'Acrobat Distiller 5.0.5 for Macintosh (via http://big.faceless.org/products/pdf?version=2.8.3)', 'creator': 'FrameMaker 7.0', 'creationdate': '2006-01-28T12:41:19+00:00', 'author': 'DWest', 'moddate': '2018-09-04T09:14:41+00:00', 'title': 'R0401H_pdf.fm', 'source': 'data/pdf/What Makes a Leader - Daniel Goleman HBR 2004_b6e637.pdf', 'total_pages': 12, 'page': 0, 'page_label': '1', 'source_file': 'What Makes a Leader - Daniel Goleman HBR 2004_b6e637.pdf', 'file_type': 'pdf'}...
